# Случайный лес — практическое задание

В этом ноутбуке рассматривается метод **случайного леса** на примере набора данных **Breast Cancer Wisconsin (Diagnostic) Data** с Kaggle.  
Ноутбук рассчитан на выполнение в **Google Colab**.

## Теоретическое введение

### 1. Постановка задачи классификации

Пусть имеется обучающая выборка
$$
\{(x_i, y_i)\}_{i=1}^n, \qquad x_i \in \mathbb{R}^d,\quad y_i \in \{0,1\}.
$$

Здесь:
- $x_i$ — вектор признаков объекта;
- $y_i$ — метка класса.

Требуется построить алгоритм, который по признакам нового объекта предсказывает его класс.

### 2. Идея ансамбля моделей

Одна отдельная модель может оказаться слишком чувствительной к особенностям обучающей выборки.  
Идея ансамблевых методов состоит в том, чтобы построить **несколько моделей**, а затем объединить их ответы.

Если отдельные модели достаточно разнообразны и делают ошибки в разных местах, то агрегированное решение часто оказывается более устойчивым и точным.

### 3. Случайный лес

**Случайный лес** (*Random Forest*) — это ансамбль деревьев решений.  
Каждое дерево обучается не на всей выборке целиком, а на её случайной модификации. Кроме того, при каждом разбиении вершины дерево рассматривает не все признаки сразу, а только случайно выбранное их подмножество.

Идея метода состоит из двух источников случайности:

1. **Бутстреп-выборки**  
   Для каждого дерева из исходной обучающей выборки случайно выбирается новая выборка того же размера с возвращением.

2. **Случайный выбор признаков**  
   При построении каждого разбиения дерево ищет лучший порог не среди всех признаков, а только среди случайно выбранного подмножества.

Это уменьшает коррелированность деревьев и делает ансамбль более эффективным.

### 4. Математическая схема метода

Пусть построено $B$ деревьев
$$
T_1(x), T_2(x), \dots, T_B(x).
$$

Тогда для задачи классификации итоговый ответ ансамбля задаётся правилом большинства:
$$
\hat{y}(x) = \operatorname{mode}\bigl(T_1(x), T_2(x), \dots, T_B(x)\bigr).
$$

Если модель умеет выдавать вероятности классов, то часто используется усреднение вероятностей:
$$
\hat{p}(x) = \frac{1}{B}\sum_{b=1}^B p_b(x),
$$
после чего класс выбирается по максимальной вероятности.

### 5. Почему случайный лес работает лучше одного дерева

Отдельное дерево решений:
- легко интерпретируется;
- может хорошо подстроиться под данные;
- но часто склонно к переобучению.

Случайный лес уменьшает разброс предсказаний за счёт усреднения по многим деревьям.  
Если записать ошибку как сочетание смещения и разброса, то усреднение большого числа слабокоррелированных деревьев в первую очередь уменьшает именно **variance**.

### 6. Основные параметры модели

У `RandomForestClassifier` особенно важны параметры:
- `n_estimators` — число деревьев в лесу;
- `max_depth` — максимальная глубина каждого дерева;
- `min_samples_split` — минимальное число объектов для разбиения вершины;
- `min_samples_leaf` — минимальное число объектов в листе;
- `max_features` — число признаков, случайно рассматриваемых при каждом разбиении.

### 7. Важность признаков

Случайный лес позволяет оценивать **важность признаков**.  
Интуитивно более важными считаются те признаки, которые чаще участвуют в полезных разбиениях и сильнее уменьшают неопределённость в деревьях ансамбля.

Это удобно для интерпретации и предварительного анализа данных, хотя такие оценки нужно трактовать осторожно.

## Набор данных

Для работы используется набор данных о диагностике опухолей молочной железы:  
[https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data](https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data).

В этом наборе данных:
- `diagnosis` — целевая переменная;
- значение `M` означает **malignant** (злокачественная опухоль);
- значение `B` означает **benign** (доброкачественная опухоль).

Также в таблице обычно присутствуют:
- столбец `id`, который не является информативным признаком для модели;
- столбец `Unnamed: 32`, часто полностью пустой и подлежащий удалению.

Большинство остальных признаков являются числовыми характеристиками формы и структуры клеточных ядер, например:
- `radius_mean`,
- `texture_mean`,
- `perimeter_mean`,
- `area_mean`,
- `smoothness_mean`,
- `compactness_mean` и др.

В задаче бинарной классификации удобно преобразовать целевую переменную `diagnosis` к виду:
- `M -> 1`,
- `B -> 0`.

## Как загрузить данные с Kaggle прямо в Google Colab

Выполните предварительную настройку Kaggle и Colab (делается один раз).
1. Откройте https://www.kaggle.com, зайдите в свой профиль, затем в настройках профиля найдите раздел **API Tokens**.
2. Создайте токен и сразу же скопируйте его в текстовый документ на своём локальном компьютере.
3. Там же посмотрите, под каким именем пользователя (`username`) вы работаете в Kaggle. Скопируйте его в тот же документ.
4. В Colab откройте левую панель, вкладку **Secrets**.
5. Создайте два секрета:
- `KAGGLE_USERNAME` : вставьте username
- `KAGGLE_KEY`      : вставьте token

Не забудьте установить переключатель **Доступ из блокнотов**.

Далее следуйте инструкциям в коде ниже:


In [ ]:
# ШАГ 1. Установка Kaggle API
!pip -q install kaggle

# ШАГ 2. Чтение токена из Colab Secrets
import os
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# ШАГ 3. Проверка, что всё работает
!kaggle datasets list -s "breast cancer wisconsin" | head

# ШАГ 4. Скачивание датасета
TARGET_DIR = "/content/sci-prog-and-ml"
!mkdir -p {TARGET_DIR}
os.chdir(TARGET_DIR)

# Настройка адреса скачивания: берём из URL всё, что идёт
# после /datasets/ : https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data
DATASET = "uciml/breast-cancer-wisconsin-data"
!kaggle datasets download -d {DATASET} -p {TARGET_DIR} --unzip

# ШАГ 5. Проверка содержимого папки
print("Содержимое папки:")
for file in os.listdir(TARGET_DIR):
    print(" -", file)


## Альтернативный вариант — скачивание файла вручную

- создайте у себя на Google-диске папку `sci-prog-and-ml`;
- скачайте файл данных вручную и поместите его в эту папку на Google-диске;
- подмонтируйте Google-диск к среде Colab, выполнив код ниже;

После выполнения этих инструкций папка `sci-prog-and-ml` будет сделана рабочей, и из неё уже можно будет открывать файл при помощи `df = pd.read_csv(file_name)`.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/sci-prog-and-ml')
print("Содержимое текущей папки:")
for file in os.listdir('.'):
    print(" -", file)


## Задание

### Часть 1. Загрузка и первичный анализ данных
1. Загрузите файл с данными в `pandas.DataFrame`.
2. Выведите первые строки таблицы.
3. Определите:
   - размер таблицы;
   - список признаков;
   - наличие пропущенных значений;
   - распределение целевой переменной `diagnosis`.

### Часть 2. Подготовка данных
1. Удалите технические или неинформативные столбцы, если они присутствуют:
   - `id`,
   - `Unnamed: 32`.
2. Преобразуйте целевую переменную `diagnosis` к бинарному виду:
   - `M -> 1`,
   - `B -> 0`.
3. Разделите данные на признаки `X` и целевую переменную `y`.
4. При необходимости обработайте пропущенные значения.

### Часть 3. Построение модели
1. Разделите выборку на обучающую и тестовую части.
2. Создайте модель `RandomForestClassifier`.
3. Обучите модель на обучающей выборке.
4. Получите предсказания на тестовой выборке.

### Часть 4. Оценка качества
1. Вычислите `accuracy`.
2. Постройте матрицу ошибок.
3. Выведите `classification_report`.
4. Проанализируйте, какие ошибки модель делает чаще.

### Часть 5. Анализ результатов
1. Найдите важности признаков (`feature_importances_`).
2. Отсортируйте признаки по важности.
3. Постройте столбчатую диаграмму для наиболее важных признаков.
4. Сделайте вывод, какие характеристики сильнее всего влияют на классификацию.

### Часть 6. Исследование параметров
1. Измените число деревьев `n_estimators` и сравните качество модели.
2. Попробуйте разные значения `max_depth`.
3. Обсудите, как параметры влияют на устойчивость и качество модели.


## Подсказки

### Основные библиотеки
```python
import os
import glob
import pandas as pd
import numpy as np
```

### Загрузка данных

После распаковки датасета в рабочей папке обычно появляется CSV-файл с медицинскими данными.
Чтобы код не зависел от точного имени файла, удобно автоматически найти первый подходящий CSV-файл:

```python
csv_files = glob.glob("*.csv")
print(csv_files)

df = pd.read_csv(csv_files[0])
```

### Удаление технических столбцов
```python
for col in ["id", "Unnamed: 32"]:
    if col in df.columns:
        df = df.drop(columns=col)
```

### Кодирование целевой переменной
```python
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
```

### Разделение на обучающую и тестовую выборки
```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

### Модель случайного леса
```python
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)
model.fit(X_train, y_train)
```

### Оценка качества
```python
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
```

### Важности признаков
```python
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)
```


## Решение


In [ ]:
# Поместите Ваш код сюда


## Вопросы для обсуждения

1. Почему случайный лес обычно устойчивее одного дерева решений?
2. Зачем при построении леса использовать случайный выбор признаков в вершинах?
3. Почему увеличение числа деревьев обычно повышает устойчивость модели?
4. Почему важности признаков в случайном лесе полезны, но не всегда дают окончательную интерпретацию?
5. В чём практическое отличие между одним деревом решений и ансамблем деревьев?


## Дополнительные задания

### Задание 1
Измените параметр `n_estimators` и сравните качество моделей, например при значениях:
- `50`,
- `100`,
- `300`,
- `500`.

### Задание 2
Попробуйте разные ограничения глубины:
- `max_depth=3`,
- `max_depth=5`,
- `max_depth=8`,
- `max_depth=None`.

Сравните результаты.

### Задание 3
Измените параметр `max_features` и посмотрите, как это влияет на качество и важности признаков.

### Задание 4
Сравните случайный лес с одним деревом решений на этом же наборе данных. Обсудите, какая модель лучше интерпретируется и какая показывает лучшее качество.
